In [3]:
import pandas as pd
import numpy as np

df1 = pd.read_csv('/content/dataset-1.csv')
df2 = pd.read_csv('/content/dataset-2.csv')

print('Dataset 1 Shape:', df1.shape)
print('Dataset 2 Shape:', df2.shape)

print('\nDataset 1 Columns:', df1.columns.tolist())
print('Dataset 2 Columns:', df2.columns.tolist())

print('\nDataset 1 Head:')
display(df1.head())
print('\nDataset 2 Head:')
display(df2.head())

Dataset 1 Shape: (5634, 21)
Dataset 2 Shape: (3207, 12)

Dataset 1 Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
Dataset 2 Columns: ['brand', 'model', 'model_year', 'milage', 'fuel_type', 'engine', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title', 'price']

Dataset 1 Head:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,4950-BDEUX,Male,0,No,No,35,No,No phone service,DSL,No,...,Yes,No,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65,False
1,7993-NQLJE,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,75.10,1151.55,False
2,7321-ZNSLA,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,...,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35,False
3,4922-CVPDX,Female,0,Yes,No,26,Yes,No,DSL,No,...,Yes,No,Yes,Yes,Two year,Yes,Credit card (automatic),73.50,1905.7,False
4,2903-YYTBW,Male,0,Yes,Yes,1,Yes,No,DSL,No,...,No,No,No,No,Month-to-month,No,Electronic check,44.55,44.55,False



Dataset 2 Head:


,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,BMW,M235 i,2015,"81,000 mi.",Gasoline,320.0HP 3.0L Straight 6 Cylinder Engine Gasoli...,6-Speed M/T,White,Red,None reported,Yes,"$27,890"
1,BMW,750 i,2017,"24,650 mi.",Gasoline,445.0HP 4.4L 8 Cylinder Engine Gasoline Fuel,A/T,Silver,Black,None reported,Yes,"$50,000"
2,Aston,Martin V8 Vantage Base,2008,"62,378 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,M/T,Red,Beige,At least 1 accident or damage reported,Yes,"$33,995"
3,INFINITI,QX80 Luxe,2022,"49,860 mi.",Gasoline,5.6L V8 32V GDI DOHC,Automatic,Black Obsidian,Graphite,At least 1 accident or damage reported,Yes,"$47,645"
4,Subaru,BRZ Premium,2023,"5,800 mi.",Gasoline,228.0HP 2.4L 4 Cylinder Engine Gasoline Fuel,6-Speed M/T,Black,Black,None reported,Yes,"$29,990"


In [4]:
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor


raw_df = pd.read_csv('/content/dataset-2.csv')

def clean_target(price_str):
    if pd.isna(price_str):
        return np.nan
    if isinstance(price_str, (int, float)):
        return float(price_str)
    cleaned = re.sub(r'[^0-9.]', '', str(price_str))
    return float(cleaned) if cleaned else np.nan

def clean_mileage(mileage_str):
    if pd.isna(mileage_str):
        return np.nan
    if isinstance(mileage_str, (int, float)):
        return float(mileage_str)
    cleaned = re.sub(r'[^0-9]', '', str(mileage_str))
    return float(cleaned) if cleaned else np.nan


raw_df['price'] = raw_df['price'].apply(clean_target)
raw_df['milage'] = raw_df['milage'].apply(clean_mileage)
raw_df['accident'] = raw_df['accident'].fillna('None reported')
raw_df['clean_title'] = raw_df['clean_title'].fillna('No')

raw_df = raw_df.dropna(subset=['price'])


def prepare_features(df):
    df_out = df.copy()
    df_out['milage'] = df_out['milage'].apply(clean_mileage)
    df_out['accident'] = df_out['accident'].fillna('None reported')
    df_out['clean_title'] = df_out['clean_title'].fillna('No')


    def extract_hp(engine_str):
        if pd.isna(engine_str):
            return np.nan
        match = re.search(r'([0-9.]+)\s*HP', str(engine_str), re.IGNORECASE)
        return float(match.group(1)) if match else np.nan

    df_out['hp'] = df_out['engine'].apply(extract_hp)
    return df_out

def remove_bad_training_rows(df):

    df_clean = df[(df['price'] > 500) & (df['price'] < 300000)].copy()
    return df_clean


X = raw_df.drop(columns=['price'])
y = raw_df['price']

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.15, random_state=42)

train_df = X_train.copy()
train_df['price'] = y_train
train_df = remove_bad_training_rows(train_df)
X_train = train_df.drop(columns=['price'])
y_train = train_df['price']


X_train_prep = prepare_features(X_train)
X_val_prep = prepare_features(X_val)
X_test_prep = prepare_features(X_test)


num_features = ['model_year', 'milage', 'hp']
cat_features = ['brand', 'fuel_type', 'transmission', 'accident', 'clean_title']

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(max_iter=200, random_state=42))
])

y_train_log = np.log1p(y_train)
model_pipeline.fit(X_train_prep, y_train_log)


def evaluate(model, X_eval, y_true):
    pred_log = model.predict(X_eval)
    pred = np.expm1(pred_log)

    pred = np.clip(pred, 0, None)

    mae = mean_absolute_error(y_true, pred)
    rmse = np.sqrt(mean_squared_error(y_true, pred))
    r2 = r2_score(y_true, pred)
    return mae, rmse, r2, pred


mae_t, rmse_t, r2_t, _ = evaluate(model_pipeline, X_train_prep, y_train)
mae_v, rmse_v, r2_v, _ = evaluate(model_pipeline, X_val_prep, y_val)
mae_te, rmse_te, r2_te, _ = evaluate(model_pipeline, X_test_prep, y_test)

print(f'Train      -> MAE: {mae_t:.2f} | RMSE: {rmse_t:.2f} | R2: {r2_t:.4f}')
print(f'Validation -> MAE: {mae_v:.2f} | RMSE: {rmse_v:.2f} | R2: {r2_v:.4f}')
print(f'Test       -> MAE: {mae_te:.2f} | RMSE: {rmse_te:.2f} | R2: {r2_te:.4f}')

Train      -> MAE: 6111.54 | RMSE: 14252.17 | R2: 0.8520
Validation -> MAE: 12289.60 | RMSE: 34745.09 | R2: 0.5254
Test       -> MAE: 12924.55 | RMSE: 40839.30 | R2: 0.4564
